# 🚀 Zalo AI Challenge: DataMate - Colab & Kaggle GPU Ultra-Fast Runner

Notebook này hướng dẫn tối ưu hóa và chạy toàn bộ pipeline của dự án **DataMate (Zalo AI Challenge)** trên cả môi trường **Google Colab (GPU T4/A100)** và **Kaggle Notebooks (GPU T4 x 2 / P100)**.

> ⚠️ **CẤU HÌNH:**
> - **Google Colab:** Vào **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU**.
> - **Kaggle:** Ở thanh Settings bên phải: Chọn **Accelerator: GPU T4 x 2** (hoặc P100) VÀ BẮT BUỘC gạt **Internet on**.

### 1. Kiểm tra GPU
Nếu thấy hiển thị bảng NVIDIA-SMI với card **Tesla T4 (15360MiB VRAM)** là bạn đã sẵn sàng.

In [ ]:
!nvidia-smi

### 2. Thiết lập mã nguồn dự án
Chọn một trong hai cách bên dưới:
- **Cách A:** Clone từ GitHub (nếu bạn đã đẩy code lên GitHub cá nhân)
- **Cách B:** Giải nén từ file `Zalo-ai.zip` được tải lên Colab

In [ ]:
import os
REPO_URL = "https://github.com/Drazzct/Zalo-ai.git"

if os.path.exists("/kaggle"):
    TARGET_DIR = "/kaggle/working/Zalo-ai"
elif os.path.exists("/content"):
    TARGET_DIR = "/content/Zalo-ai"
else:
    TARGET_DIR = os.getcwd()

if not os.path.exists(TARGET_DIR) or not os.path.exists(os.path.join(TARGET_DIR, ".git")):
    print(f"📦 Đang clone mã nguồn mới nhất từ {REPO_URL} vào {TARGET_DIR}...")
    !git clone {REPO_URL} {TARGET_DIR}
else:
    print(f"🔄 Thư mục {TARGET_DIR} đã tồn tại, đang tự động đồng bộ (git pull) bản mới nhất...")
    %cd {TARGET_DIR}
    !git fetch origin main && git reset --hard origin/main
%cd {TARGET_DIR}

print("✅ Thư mục làm việc hiện tại:", os.getcwd())
!git log -1 --oneline


### 3. Cài đặt các thư viện Python cần thiết

In [ ]:
!pip install -q pandas duckdb openpyxl pydantic pytest requests openai

### 4. Cài đặt & Khởi động Ollama (Tối ưu hóa GPU & Chạy song song 4 luồng)
Cấu hình `OLLAMA_NUM_PARALLEL=4` và `OLLAMA_FLASH_ATTENTION=1` giúp T4 GPU xử lý đồng thời 4 câu hỏi cùng lúc.

In [ ]:
# 1. Cài đặt Ollama Linux (kèm zstd - công cụ giải nén bắt buộc)
import subprocess, os, time

if subprocess.run(["which", "ollama"], capture_output=True).returncode != 0:
    print("📦 Đang kiểm tra zstd (công cụ giải nén bắt buộc)...\n")
    !which zstd > /dev/null || (apt-get update -qq && apt-get install -y -qq zstd || sudo apt-get update -qq && sudo apt-get install -y -qq zstd)
    print("\n📦 Đang tải và cài đặt Ollama...\n")
    !curl -fsSL https://ollama.com/install.sh | sh
else:
    print("✅ Ollama đã được cài đặt từ trước.")

# 2. Khởi chạy Ollama daemon với biến môi trường GPU
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0,1" if os.path.exists("/kaggle") else "0"
env["OLLAMA_NUM_PARALLEL"] = "2"
env["OLLAMA_FLASH_ATTENTION"] = "1"

print("Đang khởi động Ollama daemon...")
with open("ollama.log", "w") as f:
    ollama_process = subprocess.Popen(["ollama", "serve"], env=env, stdout=f, stderr=f)
time.sleep(5) # Đợi daemon sẵn sàng

# 3. Tải model Qwen2.5-Coder 7B (Model 7B Q4_K_M nặng ~4.7GB, vừa vặn 100% trong VRAM T4 15GB)
!ollama pull qwen2.5-coder:7b
!ollama ps

### 4.1. Kiểm tra xác nhận Model đã nạp vào GPU (Bắt buộc)
Cột **PROCESSOR** trong `ollama ps` phải báo **100% GPU**. Nếu báo 100% CPU tức là Colab chưa nhận CUDA!

In [ ]:
# Khởi động model một lần để đưa vào VRAM
!ollama run qwen2.5-coder:7b "print('ready')" --verbose

# Kiểm tra bảng trạng thái nạp VRAM
!ollama ps

### 5. Chạy bộ kiểm thử (Unit Tests)
Đảm bảo toàn bộ 36 tests nghiệp vụ, calendar, sandbox, abstain gate đều PASS 100%.

In [ ]:
!pytest tests/ -v

### 6. Chạy Benchmark SIÊU TỐC với `--workers 4`
Khi bật `--workers 4`, 4 câu hỏi được gửi song song đến Ollama GPU:
- Trước khi tối ưu: ~60 giây / câu (10 câu = ~10 phút).
- Sau khi tối ưu GPU + 4 workers: **chỉ ~30 - 45 giây cho cả 10 câu** (Nhanh gấp **15 - 20 lần**)!

In [ ]:
# Chạy thử nghiệm trên 10 câu đầu tiên của tập train với 4 luồng song song:
!python scripts/run_benchmark.py \
  --split-dir train \
  --model-backend ollama \
  --model-name qwen2.5-coder:7b \
  --workers 4 \
  --limit 10 \
  --evaluate

### 7. (Tùy chọn) Chạy bằng vLLM Engine - Đỉnh cao tốc độ (< 20s cho 10 câu)
vLLM sử dụng PagedAttention và Continuous Batching chuyên dụng cho GPU server, nhanh gấp 2-3 lần cả Ollama GPU.

In [ ]:
# Bỏ ghi chú các dòng dưới nếu bạn muốn thử nghiệm vLLM:
# !pip install -q vllm
# import subprocess, time
# vllm_proc = subprocess.Popen([
#     "vllm", "serve", "Qwen/Qwen2.5-Coder-7B-Instruct-GPTQ-Int4",
#     "--port", "8000",
#     "--max-model-len", "4096",
#     "--gpu-memory-utilization", "0.85"
# ])
# time.sleep(20)
# !python scripts/run_benchmark.py \
#   --split-dir train \
#   --model-backend vllm \
#   --model-name Qwen/Qwen2.5-Coder-7B-Instruct-GPTQ-Int4 \
#   --api-url http://127.0.0.1:8000 \
#   --workers 4 \
#   --limit 10 \
#   --evaluate

### 8. Tải file kết quả `submission.jsonl` và Log về máy tính

In [ ]:
from google.colab import files
import os

if os.path.exists('submission.jsonl'):
    print("Đang tải file submission.jsonl về...")
    files.download('submission.jsonl')
else:
    print("Chưa có file submission.jsonl")